# 22 · Boundary Geometry in Local Feature Space

This notebook studies the **geometry of the separating boundary** between ensembles in the local feature space built in earlier notebooks.

## Core question

Earlier notebooks showed that:
- zeta and GUE are both easy to separate from Poisson
- zeta and GUE are much harder to separate from each other
- these patterns are stable across height and transfer across height

Now we ask:

> What does the decision boundary itself look like?

We examine:
1. zeta vs GUE
2. zeta vs Poisson
3. GUE vs Poisson

using the compact 3-feature set:
- entropy
- unevenness
- ratio_mean

## What this notebook measures

For each binary task, we compute:
- PCA embedding
- logistic-regression boundary in PCA space
- decision score histograms
- distance-to-boundary distributions
- a simple height-block view of boundary margin

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 1800
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Minimal feature set

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def build_feature_matrix(gaps, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    X = np.array([
        [window_entropy(w), unevenness(w), ratio_mean(w)]
        for w in windows_n
    ], dtype=float)
    return windows_n, X

## PCA helpers

In [ ]:
def fit_pca_2d(X):
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std[std == 0] = 1.0
    Xs = (X - mean) / std
    Xc = Xs - Xs.mean(axis=0)

    cov = np.cov(Xc, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    comps = eigvecs[:, :2]
    Z = Xc @ comps
    explained = eigvals[:2] / eigvals.sum()
    return {
        "mean": mean,
        "std": std,
        "comps": comps,
        "explained": explained,
        "Z": Z,
    }

## Lightweight logistic regression

This avoids external dependencies and is enough for boundary-shape analysis.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])

    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad

    return w

def predict_proba_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return sigmoid(Xb @ w)

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_logistic(X, w, threshold=0.5):
    return (predict_proba_logistic(X, w) >= threshold).astype(int)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

## Prepare one common comparison set

Use the first 700 zeta/Poisson gaps and a matched GUE prefix.

In [ ]:
zeta_base = zeta_gaps_full[:700]
poisson_base = poisson_gaps_full[:700]
gue_base = gue_gaps_full[:950]

_, zeta_X = build_feature_matrix(zeta_base, k=5)
_, poisson_X = build_feature_matrix(poisson_base, k=5)
_, gue_X = build_feature_matrix(gue_base, k=5)

m = min(len(zeta_X), len(poisson_X), len(gue_X))
zeta_X = zeta_X[:m]
poisson_X = poisson_X[:m]
gue_X = gue_X[:m]

zeta_X.shape, poisson_X.shape, gue_X.shape

## Task helper

For a pair of ensembles:
- fit PCA
- fit logistic boundary in PCA space
- compute decision scores and distances to boundary

In [ ]:
def run_pairwise_boundary_task(name, X0, X1):
    X = np.vstack([X0, X1])
    y = np.array([0] * len(X0) + [1] * len(X1))

    pca_model = fit_pca_2d(X)
    Z = pca_model["Z"]
    Z0 = Z[:len(X0)]
    Z1 = Z[len(X0):]

    w = fit_logistic_regression(Z, y, lr=0.15, n_steps=3000, reg=1e-4)
    scores = decision_function_logistic(Z, w)
    probs = predict_proba_logistic(Z, w)
    preds = (probs >= 0.5).astype(int)
    acc = accuracy(y, preds)

    scores0 = scores[:len(X0)]
    scores1 = scores[len(X0):]

    normw = np.linalg.norm(w[1:]) if np.linalg.norm(w[1:]) > 1e-12 else 1.0
    dists0 = np.abs(scores0) / normw
    dists1 = np.abs(scores1) / normw

    return {
        "name": name,
        "Z0": Z0,
        "Z1": Z1,
        "y": y,
        "scores0": scores0,
        "scores1": scores1,
        "dists0": dists0,
        "dists1": dists1,
        "weights": w,
        "accuracy": acc,
        "pca_explained": pca_model["explained"],
    }

## Run the three main tasks

In [ ]:
task_zeta_gue = run_pairwise_boundary_task("zeta vs GUE", zeta_X, gue_X)
task_zeta_pois = run_pairwise_boundary_task("zeta vs Poisson", zeta_X, poisson_X)
task_gue_pois = run_pairwise_boundary_task("GUE vs Poisson", gue_X, poisson_X)

task_zeta_gue["accuracy"], task_zeta_pois["accuracy"], task_gue_pois["accuracy"]

## PCA embeddings with boundary lines

In [ ]:
def plot_task_with_boundary(ax, task, label0, label1):
    Z0 = task["Z0"]
    Z1 = task["Z1"]
    w = task["weights"]
    expl = task["pca_explained"]

    ax.scatter(Z0[:,0], Z0[:,1], s=10, alpha=0.25, label=label0)
    ax.scatter(Z1[:,0], Z1[:,1], s=10, alpha=0.25, label=label1)

    xmin = min(Z0[:,0].min(), Z1[:,0].min())
    xmax = max(Z0[:,0].max(), Z1[:,0].max())
    xs = np.linspace(xmin, xmax, 200)

    if abs(w[2]) > 1e-10:
        ys = -(w[0] + w[1]*xs) / w[2]
        ax.plot(xs, ys, linewidth=2)
    else:
        x0 = -w[0] / w[1]
        ax.axvline(x0, linewidth=2)

    ax.set_title(f"{task['name']} · acc={task['accuracy']:.3f}")
    ax.set_xlabel(f"PC1 ({expl[0]*100:.1f}% var)")
    ax.set_ylabel(f"PC2 ({expl[1]*100:.1f}% var)")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
plot_task_with_boundary(axes[0], task_zeta_gue, "zeta", "GUE")
plot_task_with_boundary(axes[1], task_zeta_pois, "zeta", "Poisson")
plot_task_with_boundary(axes[2], task_gue_pois, "GUE", "Poisson")
axes[0].legend()
plt.tight_layout()
plt.show()

## Decision-score histograms

Scores near zero indicate ambiguity.  
Wide separation means an easier task.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)

for ax, task, l0, l1 in [
    (axes[0], task_zeta_gue, "zeta", "GUE"),
    (axes[1], task_zeta_pois, "zeta", "Poisson"),
    (axes[2], task_gue_pois, "GUE", "Poisson"),
]:
    ax.hist(task["scores0"], bins=25, alpha=0.6, density=True, label=l0)
    ax.hist(task["scores1"], bins=25, alpha=0.6, density=True, label=l1)
    ax.axvline(0.0, linestyle="--", linewidth=1)
    ax.set_title(task["name"])
    ax.set_xlabel("decision score")

axes[0].set_ylabel("density")
axes[0].legend()
plt.tight_layout()
plt.show()

## Distance-to-boundary distributions

Small distance means the point lies near the separating boundary.  
If both ensembles are close to the boundary, the task is intrinsically weak.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)

for ax, task, l0, l1 in [
    (axes[0], task_zeta_gue, "zeta", "GUE"),
    (axes[1], task_zeta_pois, "zeta", "Poisson"),
    (axes[2], task_gue_pois, "GUE", "Poisson"),
]:
    ax.hist(task["dists0"], bins=25, alpha=0.6, density=True, label=l0)
    ax.hist(task["dists1"], bins=25, alpha=0.6, density=True, label=l1)
    ax.set_title(task["name"])
    ax.set_xlabel("distance to boundary")

axes[0].set_ylabel("density")
axes[0].legend()
plt.tight_layout()
plt.show()

## Boundary-margin summary

We compare mean distances to the boundary.

In [ ]:
margin_summary = {
    "zeta_vs_GUE": {
        "mean_dist_0": float(np.mean(task_zeta_gue["dists0"])),
        "mean_dist_1": float(np.mean(task_zeta_gue["dists1"])),
        "accuracy": task_zeta_gue["accuracy"],
    },
    "zeta_vs_Poisson": {
        "mean_dist_0": float(np.mean(task_zeta_pois["dists0"])),
        "mean_dist_1": float(np.mean(task_zeta_pois["dists1"])),
        "accuracy": task_zeta_pois["accuracy"],
    },
    "GUE_vs_Poisson": {
        "mean_dist_0": float(np.mean(task_gue_pois["dists0"])),
        "mean_dist_1": float(np.mean(task_gue_pois["dists1"])),
        "accuracy": task_gue_pois["accuracy"],
    },
}
margin_summary

## Height-block boundary margin for zeta vs GUE

Track how the mean distance-to-boundary changes across height for the hardest task.

In [ ]:
block_size = 300
block_starts = [0, 300, 600, 900, 1200]
block_labels = [f"{s+1}-{s+block_size}" for s in block_starts]

zeta_gue_height = []

for s, label in zip(block_starts, block_labels):
    zeta_block = zeta_gaps_full[s:s + block_size]
    _, zeta_block_X = build_feature_matrix(zeta_block, k=5)

    m_block = min(len(zeta_block_X), len(gue_X))
    task = run_pairwise_boundary_task("zeta vs GUE", zeta_block_X[:m_block], gue_X[:m_block])

    zeta_gue_height.append({
        "block": label,
        "accuracy": task["accuracy"],
        "mean_zeta_dist": float(np.mean(task["dists0"])),
        "mean_gue_dist": float(np.mean(task["dists1"])),
    })

zeta_gue_height

In [ ]:
x = np.arange(len(block_labels))
zeta_d = [r["mean_zeta_dist"] for r in zeta_gue_height]
gue_d = [r["mean_gue_dist"] for r in zeta_gue_height]
accs = [r["accuracy"] for r in zeta_gue_height]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

axes[0].plot(block_labels, zeta_d, marker="o", label="zeta distance")
axes[0].plot(block_labels, gue_d, marker="o", label="GUE distance")
axes[0].set_ylabel("mean distance to boundary")
axes[0].set_title("zeta vs GUE margin across height")
axes[0].legend()

axes[1].plot(block_labels, accs, marker="o")
axes[1].set_ylabel("classification accuracy")
axes[1].set_title("zeta vs GUE accuracy across height")

plt.tight_layout()
plt.show()

## Compact summary

In [ ]:
summary = {
    "task_accuracies": {
        "zeta_vs_GUE": task_zeta_gue["accuracy"],
        "zeta_vs_Poisson": task_zeta_pois["accuracy"],
        "GUE_vs_Poisson": task_gue_pois["accuracy"],
    },
    "margin_summary": margin_summary,
    "zeta_gue_height": zeta_gue_height,
}
summary

## Notes

- If zeta vs GUE has lower accuracy and smaller distances to the boundary than the Poisson tasks, that supports the idea that the shared local structure is intrinsically hard to separate, not just poorly measured.
- The boundary analysis here uses PCA-reduced coordinates and a simple logistic model. That is enough to study margin geometry, even though it is not the only possible classifier.
- The height-block analysis is included only for the hardest task; later notebooks could extend the same analysis to the Poisson comparisons if needed.

## Next directions

A natural next notebook could do one of these:

1. compare linear vs nonlinear boundaries explicitly  
2. train on GUE vs Poisson and test directly on zeta vs GUE confidence scores  
3. bootstrap the boundary-margin statistics  
4. study how decision geometry changes under alternative feature sets